### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="polish_companies_bankruptcy",
    dataset_year="2010",
    domain_str="finance",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5F600",
    download_description="""
We download the data from the UCI repository and uzip it to a predefined folder.

mkdir -p local-data-warehouse/polish_companies_bankruptcy/ && wget -P local-data-warehouse/polish_companies_bankruptcy/ https://archive.ics.uci.edu/static/public/365/polish+companies+bankruptcy+data.zip && unzip local-data-warehouse/polish_companies_bankruptcy/polish+companies+bankruptcy+data.zip -d local-data-warehouse/polish_companies_bankruptcy/ && rm local-data-warehouse/polish_companies_bankruptcy/polish+companies+bankruptcy+data.zip
""",
    # References
    academic_reference_bibtex="""@article{zikeba2016ensemble,
  title={Ensemble boosted trees with synthetic features generation in application to bankruptcy prediction},
  author={Zi{\k{e}}ba, Maciej and Tomczak, Sebastian K and Tomczak, Jakub M},
  journal={Expert systems with applications},
  volume={58},
  pages={93--101},
  year={2016},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="zikeba2016ensemble",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- We only use data from year 5 (5year.arff), because it is the newest data and the target is bankruptcy status after only 1 year.
- We created semantically meaningful feature names.
- We removed duplicates.
- Anomaly: the data contains a lot of features created by feature engineering.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="company_bankrupt",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="company_bankrupt",
)

## Preprocessing

In [ ]:
import arff
import pandas as pd

with open(dataset_mold.path /"5year.arff") as f:
        data = arff.load(f)

df = pd.DataFrame(data["data"], columns=[x[0] for x in data["attributes"]])

target_feature = task_mold.target_column_name
df.columns = [
    "net_profit_to_total_assets",
    "total_liabilities_to_total_assets",
    "working_capital_to_total_assets",
    "current_assets_to_short_term_liabilities",
    "liquidity_days_ratio",
    "retained_earnings_to_total_assets",
    "ebit_to_total_assets",
    "book_value_equity_to_total_liabilities",
    "sales_to_total_assets",
    "equity_to_total_assets",
    "extended_profit_to_total_assets",
    "gross_profit_to_short_term_liabilities",
    "gross_profit_plus_depreciation_to_sales",
    "gross_profit_plus_interest_to_total_assets",
    "liabilities_days_ratio",
    "gross_profit_plus_depreciation_to_total_liabilities",
    "total_assets_to_total_liabilities",
    "gross_profit_to_total_assets",
    "gross_profit_to_sales",
    "inventory_days_ratio",
    "sales_growth_ratio",
    "operating_profit_to_total_assets",
    "net_profit_to_sales",
    "three_year_gross_profit_to_total_assets",
    "equity_minus_share_capital_to_total_assets",
    "net_profit_plus_depreciation_to_total_liabilities",
    "operating_profit_to_financial_expenses",
    "working_capital_to_fixed_assets",
    "log_total_assets",
    "net_liabilities_to_sales",
    "gross_profit_plus_interest_to_sales",
    "current_liabilities_days_ratio",
    "operating_expenses_to_short_term_liabilities",
    "operating_expenses_to_total_liabilities",
    "sales_profit_to_total_assets",
    "total_sales_to_total_assets",
    "current_assets_minus_inventories_to_long_term_liabilities",
    "constant_capital_to_total_assets",
    "sales_profit_to_sales",
    "liquid_assets_to_short_term_liabilities",
    "liabilities_to_adjusted_operating_profit",
    "operating_profit_to_sales",
    "receivables_plus_inventory_turnover_days",
    "receivables_days_ratio",
    "net_profit_to_inventory",
    "current_assets_minus_inventory_to_short_term_liabilities",
    "inventory_days_cost_ratio",
    "ebitda_to_total_assets",
    "ebitda_to_sales",
    "current_assets_to_total_liabilities",
    "short_term_liabilities_to_total_assets",
    "short_term_liabilities_days_cost_ratio",
    "equity_to_fixed_assets",
    "constant_capital_to_fixed_assets",
    "working_capital_absolute",
    "gross_margin",
    "adjusted_liquidity_ratio",
    "total_costs_to_total_sales",
    "long_term_liabilities_to_equity",
    "inventory_turnover_ratio",
    "receivables_turnover_ratio",
    "short_term_liabilities_days_ratio",
    "sales_to_short_term_liabilities",
    "sales_to_fixed_assets",
    target_feature,
]

df[target_feature] = df[target_feature].map({"1": "Yes", "0": "No"}).astype("category")

# conflicting duplicates drop (without target column)
df = df.drop_duplicates(subset=[c for c in df.columns if c != task_mold.target_column_name], keep=False)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()